In [1]:
import math
from collections import Counter

# ── Dataset: job roles with their required skills ──────────────
job_roles = {
    "Data Scientist":        ["Python", "Machine Learning", "SQL", "Statistics"],
    "Web Developer":         ["JavaScript", "HTML", "CSS", "React"],
    "Cloud Architect":       ["AWS", "Cloud Computing", "Docker", "Automation"],
    "ML Engineer":           ["Python", "Machine Learning", "Docker", "Automation"],
    "Backend Developer":     ["Python", "SQL", "Docker", "APIs"],
    "Data Analyst":          ["SQL", "Statistics", "Python", "Excel"],
    "DevOps Engineer":       ["Docker", "Automation", "AWS", "Linux"],
    "AI Researcher":         ["Python", "Machine Learning", "Statistics", "Research"],
}

print("✅ Dataset loaded —", len(job_roles), "job roles")

✅ Dataset loaded — 8 job roles


In [2]:
# ── Build vocabulary from all skills in the dataset ───────────
all_skills = []
for skills in job_roles.values():
    all_skills.extend(skills)

vocabulary = sorted(set(all_skills))
print("Vocabulary:", vocabulary)

# ── TF: term frequency within each role ───────────────────────
def compute_tf(skill_list):
    count = Counter(skill_list)
    total = len(skill_list)
    return {skill: count[skill] / total for skill in skill_list}

# ── IDF: log(total_docs / docs_containing_term) ───────────────
def compute_idf(roles_dict):
    N = len(roles_dict)
    idf = {}
    for term in vocabulary:
        docs_with_term = sum(1 for skills in roles_dict.values()
                           if term in skills)
        idf[term] = math.log(N / (1 + docs_with_term))
    return idf

# ── TF-IDF vector for a skill list ────────────────────────────
def tfidf_vector(skill_list, idf_scores):
    tf = compute_tf(skill_list)
    return [tf.get(term, 0) * idf_scores.get(term, 0)
            for term in vocabulary]

idf_scores = compute_idf(job_roles)
print("✅ IDF scores computed")

Vocabulary: ['APIs', 'AWS', 'Automation', 'CSS', 'Cloud Computing', 'Docker', 'Excel', 'HTML', 'JavaScript', 'Linux', 'Machine Learning', 'Python', 'React', 'Research', 'SQL', 'Statistics']
✅ IDF scores computed


In [3]:
# ── Cosine similarity: dot product / (||A|| × ||B||) ──────────
def cosine_similarity(vec_a, vec_b):
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    mag_a = math.sqrt(sum(a ** 2 for a in vec_a))
    mag_b = math.sqrt(sum(b ** 2 for b in vec_b))
    if mag_a == 0 or mag_b == 0:
        return 0.0          # cold start — zero vector
    return dot_product / (mag_a * mag_b)

print("✅ Cosine similarity function ready")

✅ Cosine similarity function ready


In [4]:
def recommend(user_skills, top_n=3):
    # STEP 1 — Ingestion: validate input
    if len(user_skills) < 3:
        raise ValueError("Enter at least 3 skills for accurate matching.")

    # STEP 2 — Scoring: compute cosine similarity for every role
    user_vec = tfidf_vector(user_skills, idf_scores)
    scores = {}
    for role, skills in job_roles.items():
        role_vec = tfidf_vector(skills, idf_scores)
        scores[role] = cosine_similarity(user_vec, role_vec)

    # STEP 3 — Sorting: highest similarity first
    sorted_roles = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # STEP 4 — Filtering: return Top-N only
    return sorted_roles[:top_n]

print("✅ Pipeline ready")

✅ Pipeline ready


In [5]:
# ── Change these skills to test different users ────────────────
user_skills = ["Python", "Cloud Computing", "Automation"]

results = recommend(user_skills, top_n=3)

print(f"\n👤 User skills: {user_skills}\n")
print("🏆 Top Job Role Recommendations")
print("-" * 35)
for i, (role, score) in enumerate(results, 1):
    bar = "█" * int(score * 20)
    print(f"{i}. {role:<22} {score:.4f}  {bar}")


👤 User skills: ['Python', 'Cloud Computing', 'Automation']

🏆 Top Job Role Recommendations
-----------------------------------
1. Cloud Architect        0.8048  ████████████████
2. ML Engineer            0.3177  ██████
3. DevOps Engineer        0.1610  ███
